# 01 — Data Analysis (EDA)

Phân tích dữ liệu thô **trước** preprocessing offline.  
Owner: **Team analysis + preprocessing offline**.

Chạy notebook này **hai lần**:
- Lần 1: `DATA_ROOT = 'data/raw'` — trên ảnh gốc.
- Lần 2: `DATA_ROOT = 'data/processed'` — sau khi chạy preprocessing offline (notebook 02).

Kết quả xuất 5 biểu đồ vào `analysis/figures/`:
1. `01_class_distribution` – phân bố box theo lớp (52 lớp VNTS)
2. `02_box_dimensions` – phân bố w, h, aspect ratio, area
3. `03_box_center_heatmap` – vị trí tâm (cx, cy) trên khung hình
4. `04_images_per_split` – số ảnh và box theo train/val/test
5. `05_tiny_box_candidates` – box kích thước < min_box_px


## 0) Bootstrap

Chạy trên local hoặc Kaggle/Colab. Trên Kaggle bật **Internet: On**.


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Tìm repo root ──────────────────────────────────────────────
REPO_URL    = 'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git'
REPO_DIR    = 'VN-Traffic-Sign-Detection'
REPO_BRANCH = 'preprocessing'  # nhánh của Team analysis

base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd):
    print('>>', cmd)
    subprocess.run(cmd, shell=True, check=True)

if (Path.cwd() / '.git').exists() and (Path.cwd() / 'configs' / 'shared.yaml').exists():
    repo_path = Path.cwd()
    print('Using the repo this notebook sits in:', repo_path)
elif not (repo_path / '.git').exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

git = f'git -C "{repo_path}"'
sh(f'{git} fetch origin {REPO_BRANCH}')
sh(f'{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print('cwd:', Path.cwd())

sh('pip install -q -e "."')
print('Bootstrap OK.')

## 1) Cấu hình

Sửa `DATA_ROOT` trỏ đến dataset. Trên Kaggle: **Add Input** rồi điền path vào `DATA_ROOT`.


In [ ]:
from pathlib import Path
import os

# ── THAY ĐỔI BIẾN NÀY ─────────────────────────────────────────
# Lần 1 (raw):       DATA_ROOT = 'data/raw'
# Lần 2 (processed): DATA_ROOT = 'data/processed'
DATA_ROOT = os.environ.get('DATA_ROOT') or 'data/raw'
# ──────────────────────────────────────────────────────────────

FIGURES_DIR = 'analysis/figures'
CLASS_TABLE = 'configs/classes.csv'
IMAGE_SIZE  = 640    # px – để tính kích thước box
MIN_BOX_PX  = 8     # ngưỡng tiny box theo shared.yaml

print(f'DATA_ROOT  : {Path(DATA_ROOT).resolve()}')
print(f'FIGURES_DIR: {Path(FIGURES_DIR).resolve()}')

## 2) Kiểm tra dataset


In [ ]:
from pathlib import Path

root = Path(DATA_ROOT).resolve()
assert root.exists(), f'[!] Không tìm thấy DATA_ROOT: {root}'

# Kiểm tra cấu trúc
img_dir = root / 'images'
lbl_dir = root / 'labels'

# Hỗ trợ cả cấu trúc flat và processed
if lbl_dir.is_dir():
    lbl_files = list(lbl_dir.rglob('*.txt'))
else:
    lbl_files = list(root.rglob('*.txt'))

if img_dir.is_dir():
    img_files = list(img_dir.rglob('*.*'))
    img_files = [f for f in img_files if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
else:
    img_files = []

print(f'Ảnh tìm thấy : {len(img_files)}')
print(f'File nhãn    : {len(lbl_files)}')
print(f'Cấu trúc thư mục {root}:')
for child in sorted(root.iterdir()):
    print(f'  {child.name}/ ' if child.is_dir() else f'  {child.name}')

## 3) Chạy EDA – Tạo 5 biểu đồ


In [ ]:
from analysis.eda import main as eda_main

eda_main([
    '--data-root',   DATA_ROOT,
    '--figures-dir', FIGURES_DIR,
    '--class-table', CLASS_TABLE,
    '--image-size',  str(IMAGE_SIZE),
    '--min-box-px',  str(MIN_BOX_PX),
])

## 4) Xem kết quả


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path(FIGURES_DIR)
figs = sorted(fig_dir.glob('*.png'))

print(f'Biểu đồ trong {fig_dir}:')
for f in figs:
    print(f'  {f.name}')

for fpath in figs:
    img = mpimg.imread(fpath)
    plt.figure(figsize=(16, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fpath.stem, fontsize=11)
    plt.tight_layout()
    plt.show()

## 5) Tóm tắt quan sát

*(Điền sau khi xem biểu đồ)*

| Biểu đồ | Quan sát chính |
|---------|----------------|
| Class distribution | |
| Box dimensions | |
| Center heatmap | |
| Images per split | |
| Tiny box candidates | |

**Kết luận**: (Mô tả class imbalance, scale phổ biến, spatial prior của biển báo, % box bị loại…)
